# Notebook 1: nnU-Net on Kaggle, from data to `stats.npz`

Runs on **Kaggle** (GPU + Internet on). Run the cells **in order, one at a time**, and read each output before the next.
This version is the **smoke test**: 5 training epochs and 5 patients. The numbers it produces are meaningless; it only checks that the pipeline runs, measures the time per epoch, and produces a tiny `stats_smoke.npz` that the laptop notebook can load.

| Step | Cells | Output |
|---|---|---|
| Setup | 1-4 | GPU checked, nnU-Net installed, repository cloned |
| Data | 5-8 | MSD Task01 downloaded, converted, preprocessed |
| Smoke training | 9-10 | 5 epochs, time per epoch |
| Extraction | 11-12 | `stats_smoke.npz` |

For the real run, change only the settings cell (trainer name, number of patients, TTA).

## Settings

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/care-reproduction.git"   # <- put your GitHub username
TRAINER = "nnUNetTrainer_5epochs"     # smoke test. Real run: nnUNetTrainer_250epochs (decide after timing)
FOLD = 0
N_CASES = 5                           # patients to extract in the smoke test. Real run: None (all held-out patients)
USE_TTA = False                       # mirroring test-time augmentation. Off for the smoke test.
OUT_NAME = "stats_smoke.npz"          # real run: stats.npz

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install nnU-Net
Kaggle already has PyTorch. This adds nnU-Net and its small dependencies (about a minute).

In [ ]:
!pip install -q nnunetv2
import importlib.metadata as md
print('nnunetv2', md.version('nnunetv2'))

## 3. Get the code from GitHub

In [ ]:
import os, subprocess
REPO = "/kaggle/working/care-reproduction"
if os.path.isdir(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", REPO_URL, REPO], capture_output=True, text=True))
print(sorted(os.listdir(REPO)))

## 4. Tell nnU-Net where things live
- raw and preprocessed data are big and temporary: `/kaggle/temp` if it exists, otherwise `/tmp` (not saved with the notebook output; check the free disk space printed below)
- trained models are small and precious: `/kaggle/working` (saved as notebook output)

In [ ]:
import os, shutil
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp/care_temp"   # big, not saved with the output
os.makedirs(TEMP, exist_ok=True)
os.environ["nnUNet_raw"] = f"{TEMP}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{TEMP}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.makedirs(os.environ[k], exist_ok=True)
    print(k, "->", os.environ[k])
print("free disk in TEMP: %.0f GB" % (shutil.disk_usage(TEMP).free / 1e9))

## 5. Download the dataset (YOU fill this in)
**Before running this cell**, do these checks yourself:
1. Open the official page of the *Medical Segmentation Decathlon* and find **Task01_BrainTumour**. Read the licence (CC-BY-SA 4.0 as far as I know; confirm it) and any conditions on use.
2. Choose a download source you trust and put its direct link in `MSD_URL`. I deliberately left it empty: I could not verify a direct link, and a download needs your confirmation.
3. Prefer a source that publishes a checksum (md5 or sha256) and compare it with the one printed by the cell.

The archive is about 7 GB; the download takes several minutes. If the link needs a login or a Google Drive confirmation, download it by another route and add it to the notebook as a Kaggle dataset instead.

In [ ]:
MSD_URL = ""    # <- direct link to Task01_BrainTumour.tar, after you checked source and licence
assert MSD_URL, "Fill in MSD_URL first (see the cell above)"
!wget -q --show-progress -O $TEMP/Task01_BrainTumour.tar "$MSD_URL"
!md5sum $TEMP/Task01_BrainTumour.tar
!tar -xf $TEMP/Task01_BrainTumour.tar -C $TEMP
!ls $TEMP/Task01_BrainTumour

## 6. Convert to nnU-Net format
The converter turns each 4D file (4 modalities) into 4 files and creates `Dataset001_BrainTumour`.

In [ ]:
!nnUNetv2_convert_MSD_dataset -i $TEMP/Task01_BrainTumour
!ls $TEMP/nnUNet_raw/Dataset001_BrainTumour

## 7. Plan and preprocess
Only the 3d_fullres configuration (the one we train). This resamples every scan to a common spacing, normalizes each modality and crops to the non-zero region. Expect 20-60 minutes.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 -c 3d_fullres --verify_dataset_integrity -npfp 4 -np 4

## 8. Look at one patient: shapes and labels
Checks the assumption that every volume has the same number of voxels, and shows which label values exist.

In [ ]:
import glob, numpy as np
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO
raw = os.environ["nnUNet_raw"] + "/Dataset001_BrainTumour"
labels = sorted(glob.glob(raw + "/labelsTr/*.nii.gz"))
print(len(labels), "labelled patients")
shapes = {}
for f in labels[:20]:
    seg, _ = SimpleITKIO().read_seg(f)
    shapes.setdefault(seg.shape, 0); shapes[seg.shape] += 1
print("shapes of the first 20 patients:", shapes)
seg, _ = SimpleITKIO().read_seg(labels[0])
print("label values in patient 0:", np.unique(seg, return_counts=True))
import json; print(json.load(open(raw + "/dataset.json"))["labels"])

## 9. Smoke training (5 epochs)
Watch the output. When it ends, the next cell prints the time per epoch. **That number decides the real plan.**

In [ ]:
!nnUNetv2_train 1 3d_fullres $FOLD -tr $TRAINER

In [ ]:
import glob, re
logs = sorted(glob.glob(os.environ["nnUNet_results"] + f"/Dataset001_BrainTumour/{TRAINER}__nnUNetPlans__3d_fullres/fold_{FOLD}/training_log_*.txt"))
times = [float(m.group(1)) for line in open(logs[-1]) for m in [re.search(r"Epoch time: ([\d.]+) s", line)] if m]
print("epoch times (s):", times)
print("mean epoch time: %.0f s" % np.mean(times[1:] if len(times) > 1 else times))
print("-> 250 epochs would take about %.1f hours" % (250 * np.mean(times[1:] if len(times) > 1 else times) / 3600))

## 10. Extract statistics for a few held-out patients
Uses `scripts/extract_stats.py` from your repository. The probabilities are never saved; each patient becomes one row of sums.

In [ ]:
model_dir = f"{os.environ['nnUNet_results']}/Dataset001_BrainTumour/{TRAINER}__nnUNetPlans__3d_fullres"
splits = f"{os.environ['nnUNet_preprocessed']}/Dataset001_BrainTumour/splits_final.json"
cmd = (f"python {REPO}/scripts/extract_stats.py --model-dir {model_dir} --dataset-dir {raw} --splits-file {splits} "
       f"--fold {FOLD} --out /kaggle/working/{OUT_NAME}")
if N_CASES: cmd += f" --n-cases {N_CASES}"
if USE_TTA: cmd += " --tta"
print(cmd)
!{cmd}

## 11. Check the output with the laptop code
The same `check_stats` that your laptop notebook uses.

In [ ]:
import sys; sys.path.insert(0, REPO)
from care import stats as S
st = S.load_stats(f"/kaggle/working/{OUT_NAME}")
print(S.check_stats(st), S.n_cases(st), "patients")
import pandas as pd
pd.DataFrame({"case": st["case_id"], "tumour voxels": st["syB"], "true ratio": S.true_ratio(st), "predicted ratio": S.pred_ratio(st),
              "|V-Bias| A": abs(S.vbias(st, "A")), "ECE A": S.ece(st, "A")})

## 12. Save and download
Use **Save Version** (top right) to keep the notebook output, then open the **Output** tab and download `stats_smoke.npz` and its `.json` log.
Put them in the `data/` folder of the laptop project and paste here what the previous cells printed: the GPU, the shapes, the label values, the epoch times and the table.

Then stop the session (session panel -> Stop Session) so the GPU quota is not consumed.